In [3]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import RectBivariateSpline
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
plt.rcParams.update({
    "figure.facecolor": "#0d1117", "axes.facecolor":  "#0d1117",
    "axes.edgecolor":   "#30363d", "axes.labelcolor": "#c9d1d9",
    "xtick.color":      "#8b949e", "ytick.color":     "#8b949e",
    "text.color":       "#c9d1d9", "grid.color":      "#21262d",
    "grid.linestyle":   "--",      "grid.linewidth":  0.5,
    "lines.linewidth":  1.2,       "font.family":     "monospace",
})

NEON_GREEN  = "#39d353"
NEON_BLUE   = "#58a6ff"
NEON_ORANGE = "#f78166"
NEON_PURPLE = "#bc8cff"
NEON_YELLOW = "#e3b341"

print("Downloading market data …")
TICKERS = ["SPY", "QQQ", "IWM", "XLF", "XLK"]
raw = yf.download(TICKERS, start="2018-01-01", auto_adjust=True, progress=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = ["_".join(col) for col in raw.columns]
close = pd.DataFrame({t: raw[f"Close_{t}"] for t in TICKERS}).dropna()

spy = close[["SPY"]].copy()
spy.columns = ["Close"]
spy["log_ret"] = np.log(spy["Close"]).diff()
for w in [5, 10, 21, 63]:
    spy[f"rv_{w}"] = spy["log_ret"].rolling(w).std() * np.sqrt(252)
spy["vix_proxy"] = spy["rv_21"] * 1.15

lambda_ = 0.94
lra = spy["log_ret"].fillna(0).values
ev = np.zeros(len(lra))
ev[0] = np.var(lra[lra != 0])
for t in range(1, len(lra)):
    ev[t] = lambda_ * ev[t-1] + (1 - lambda_) * lra[t]**2
spy["ewma_vol"] = np.sqrt(ev * 252)

def fit_garch11(r, omega=1e-6, alpha=0.09, beta=0.90, n_iter=200):
    n = len(r)
    h = np.zeros(n)
    h[0] = np.var(r)
    for _ in range(n_iter):
        for t in range(1, n):
            h[t] = omega + alpha * r[t-1]**2 + beta * h[t-1]
    return np.sqrt(h * 252)

spy["garch_vol"] = fit_garch11(spy["log_ret"].fillna(0).values)

def classify_vol_regime(vol_series, n_regimes=3):
    vov = vol_series.rolling(21).std().bfill()
    X = np.column_stack([vol_series.bfill(), vov.bfill()])
    centers = X[np.random.choice(len(X), n_regimes, replace=False)]
    labels = np.zeros(len(X), dtype=int)
    for _ in range(100):
        dists = np.array([np.linalg.norm(X - c, axis=1) for c in centers])
        labels = np.argmin(dists, axis=0)
        for k in range(n_regimes):
            mask = labels == k
            if mask.sum() > 0:
                centers[k] = X[mask].mean(axis=0)
    order = np.argsort(centers[:, 0])
    remap = {old: new for new, old in enumerate(order)}
    return pd.Series([remap[l] for l in labels], index=vol_series.index, name="regime")

spy["regime"] = classify_vol_regime(spy["ewma_vol"])

def svi_slice(k, a, b, rho, m, sigma):
    d = k - m
    return a + b * (rho * d + np.sqrt(d**2 + sigma**2))

def nelson_siegel_term(T, beta0, beta1, beta2, tau=0.5):
    x = T / tau
    return beta0 + (beta1 + beta2) * (1 - np.exp(-x)) / x - beta2 * np.exp(-x)

def build_vol_surface(S0, base_vol, regime=1):
    params = {
        0: dict(a=0.01, b=0.12, rho=-0.30, m=0.0, sigma=0.15, beta0=base_vol, beta1=-0.04, beta2=0.02),
        1: dict(a=0.02, b=0.18, rho=-0.50, m=0.0, sigma=0.20, beta0=base_vol, beta1=-0.06, beta2=0.04),
        2: dict(a=0.05, b=0.28, rho=-0.70, m=0.0, sigma=0.30, beta0=base_vol, beta1=-0.10, beta2=0.08),
    }[regime]
    p = params
    strikes    = np.linspace(0.70 * S0, 1.30 * S0, 40)
    maturities = np.array([1/12, 3/12, 6/12, 1.0, 1.5, 2.0])
    lm = np.log(strikes / S0)
    surface = np.zeros((len(maturities), len(strikes)))
    for i, T in enumerate(maturities):
        tm = nelson_siegel_term(T, p["beta0"], p["beta1"], p["beta2"])
        for j, k in enumerate(lm):
            tv = svi_slice(k, p["a"], p["b"], p["rho"], p["m"], p["sigma"])
            surface[i, j] = np.sqrt(abs(tv) / T) * (tm / p["beta0"])
    return strikes, maturities, surface

lra_all = np.log(close).diff().dropna()
crv = lra_all.rolling(21).std() * np.sqrt(252)
dispersion = (crv[["QQQ","IWM","XLF","XLK"]].mean(axis=1) - crv["SPY"]).dropna()
dispersion.name = "dispersion"
spy = spy.join(dispersion, how="left")
spy["dispersion"] = spy["dispersion"].ffill()
spy["vrp"] = spy["vix_proxy"] - spy["rv_21"]

def strategy_vrp(df, i, lb=63):
    h = df["vrp"].iloc[max(0, i-lb):i]
    if len(h) < 20: return 0.0, False
    mu, sd = h.mean(), h.std()
    v = df["vrp"].iloc[i]
    if abs(v - mu) <= 1.5 * sd: return 0.0, False
    return float(-np.sign(v - mu)), True

def strategy_regime_breakout(df, i):
    if i < 2: return 0.0, False
    rn, rp = df["regime"].iloc[i], df["regime"].iloc[i-1]
    if rn > rp and df["garch_vol"].iloc[i] > df["ewma_vol"].iloc[i]:
        return 1.0, True
    return (1.0, False) if rn == 2 else (0.0, False)

def strategy_dispersion(df, i, lb=63):
    h = df["dispersion"].iloc[max(0, i-lb):i]
    if len(h) < 20: return 0.0, False
    z = (df["dispersion"].iloc[i] - h.mean()) / (h.std() + 1e-9)
    pct = (df["ewma_vol"].iloc[:i+1] <= df["ewma_vol"].iloc[i]).mean()
    if z > 1.5 and pct < 0.30: return 1.0, True
    return 0.0, False

def strategy_skew_arb(df, i, lb=21):
    if len(df["ewma_vol"].iloc[max(0, i-lb):i]) < 5: return 0.0, False
    g, e = df["garch_vol"].iloc[i], df["ewma_vol"].iloc[i]
    rv5  = df["rv_5"].iloc[i]  if "rv_5"  in df.columns else e
    rv21 = df["rv_21"].iloc[i] if "rv_21" in df.columns else e
    if g > e * 1.10 and rv5 / (rv21 + 1e-9) < 0.80: return 1.0, True
    return 0.0, False

def fwd_proxy(p):
    mid = p[len(p)//2]
    return max((p[-1] - mid) / (mid + 1e-9), 0.0)

def corr_var_proxy(p, lo, hi):
    lr = np.diff(np.log(p + 1e-12))
    return np.sum(lr[(p[:-1] > lo) & (p[:-1] < hi)]**2)

def cliquet_proxy(p, cap=0.02):
    return float(np.sum(np.minimum(np.diff(p) / (p[:-1] + 1e-9), cap)))

def barrier_proxy(p, lo, hi):
    if np.any(p <= lo) or np.any(p >= hi): return 0.0
    return max((p[-1] - p[0]) / (p[0] + 1e-9), 0.0)

print("Running backtest …")
WB = 63
records = []
for i in range(WB, len(spy) - 1):
    wp  = spy["Close"].iloc[i-WB:i].values.astype(float)
    S0  = float(wp[-1])
    vol = float(spy["ewma_vol"].iloc[i])
    lo, hi   = 0.90 * S0, 1.10 * S0
    avg_var  = max(vol**2 / 252, 1e-10)
    exotic   = (fwd_proxy(wp) + corr_var_proxy(wp, lo, hi) / (WB * avg_var)
                + cliquet_proxy(wp) + barrier_proxy(wp, lo, hi))
    sv, cv = strategy_vrp(spy, i)
    sr, cr = strategy_regime_breakout(spy, i)
    sd, cd = strategy_dispersion(spy, i)
    ss, cs = strategy_skew_arb(spy, i)
    ret = float(spy["log_ret"].iloc[i+1])
    records.append(dict(date=spy.index[i], ret_next=ret, vol=vol,
                        regime=int(spy["regime"].iloc[i]),
                        sig_exotic=exotic,  pnl_exotic=exotic * ret,
                        cond_vrp=cv,  sig_vrp=sv,  pnl_vrp=sv * ret,
                        cond_regime=cr, sig_regime=sr, pnl_regime=sr * ret,
                        cond_disp=cd, sig_disp=sd,  pnl_disp=sd * ret,
                        cond_skew=cs, sig_skew=ss,  pnl_skew=ss * ret))

bt = pd.DataFrame(records).set_index("date")
es = bt["sig_exotic"].expanding().std().replace(0, np.nan).fillna(1)
bt["sig_exotic_z"] = (bt["sig_exotic"] - bt["sig_exotic"].expanding().mean()) / es
bt["pnl_exotic_z"] = bt["sig_exotic_z"] * bt["ret_next"]

pnl_cols = ["pnl_exotic_z","pnl_vrp","pnl_regime","pnl_disp","pnl_skew"]
bt["pnl_portfolio"] = bt[pnl_cols].mean(axis=1)
for col in pnl_cols + ["pnl_portfolio"]:
    bt[f"cum_{col}"] = bt[col].cumsum()
    bt[f"rs_{col}"]  = (bt[col].rolling(63).mean() / bt[col].rolling(63).std()) * np.sqrt(252)

def perf(s, name=""):
    p = s.dropna()
    sh = np.sqrt(252) * p.mean() / (p.std() + 1e-9)
    mdd = (p.cumsum() - p.cumsum().cummax()).min()
    print(f"  {name:<25} Sharpe={sh:+.3f}  MaxDD={mdd:.4f}  "
          f"Skew={skew(p):+.2f}  Kurt={kurtosis(p):.2f}  Hit={(p>0).mean():.2%}")

print("\n===== STRATEGY PERFORMANCE =====")
for col, nm in zip(pnl_cols + ["pnl_portfolio"],
                   ["Exotic Proxy (z-scored)","VRP Mean-Reversion","Vol Regime Breakout",
                    "Conditional Dispersion","GARCH Skew Arb","EW Portfolio"]):
    perf(bt[col], nm)

print("\nGenerating figures …")

# ── Figure 1: 3D Vol Surface by Regime ──────────────────────────────────────
fig1 = plt.figure(figsize=(20, 7))
fig1.patch.set_facecolor("#0d1117")
fig1.suptitle("SVI + Nelson–Siegel Implied Volatility Surface  ·  by Regime",
              fontsize=13, color="#c9d1d9", y=0.98)
S0r, vr = float(spy["Close"].iloc[-1]), float(spy["ewma_vol"].iloc[-1])
for reg, (lbl, cmap) in enumerate(zip(
    ["Low Vol Regime","Medium Vol Regime","High Vol Regime"],
    ["viridis","plasma","inferno"]
)):
    ax3 = fig1.add_subplot(1, 3, reg+1, projection="3d")
    ax3.set_facecolor("#0d1117")
    st, mt, sf = build_vol_surface(S0r, vr, regime=reg)
    Kg, Tg = st / S0r, mt
    sp2 = RectBivariateSpline(Tg, Kg, sf, kx=3, ky=3)
    Kf = np.linspace(Kg[0], Kg[-1], 80)
    Tf = np.linspace(Tg[0], Tg[-1], 40)
    KKf, TTf = np.meshgrid(Kf, Tf)
    sff = sp2(Tf, Kf)
    surf_plot = ax3.plot_surface(KKf, TTf, sff, cmap=cmap, alpha=0.88,
                                 linewidth=0, antialiased=True)
    ax3.contourf(KKf, TTf, sff, zdir="z", offset=sff.min()-0.01,
                 cmap=cmap, alpha=0.35, levels=12)
    ax3.set_xlabel("Moneyness K/S₀", labelpad=6, fontsize=7)
    ax3.set_ylabel("Maturity (yrs)",  labelpad=6, fontsize=7)
    ax3.set_zlabel("Implied Vol",     labelpad=6, fontsize=7)
    ax3.set_title(lbl, color="#c9d1d9", fontsize=9, pad=8)
    ax3.tick_params(labelsize=6, colors="#8b949e")
    for pane in [ax3.xaxis.pane, ax3.yaxis.pane, ax3.zaxis.pane]:
        pane.fill = False; pane.set_edgecolor("#21262d")
    ax3.view_init(elev=28, azim=-60)
    fig1.colorbar(surf_plot, ax=ax3, shrink=0.45, pad=0.08,
                  label="Impl. Vol", format="%.2f")

# ── Figure 2: Vol Surface Evolution ─────────────────────────────────────────
fig2, ax2 = plt.subplots(1, 2, figsize=(16, 6))
fig2.patch.set_facecolor("#0d1117")
fig2.suptitle("Vol Surface Evolution  ·  ATM Term Structure & Skew",
              fontsize=12, color="#c9d1d9")
sdates = spy.index[np.linspace(WB, len(spy)-1, 5).astype(int)]
cts    = [NEON_BLUE, NEON_GREEN, NEON_YELLOW, NEON_ORANGE, NEON_PURPLE]
mplot  = np.array([1/12, 3/12, 6/12, 1.0, 1.5, 2.0])
for ax in ax2: ax.set_facecolor("#0d1117")
for d, c in zip(sdates, cts):
    s0, v0, reg = float(spy.loc[d,"Close"]), float(spy.loc[d,"ewma_vol"]), int(spy.loc[d,"regime"])
    st, _, sf = build_vol_surface(s0, v0, regime=reg)
    ax2[0].plot(mplot, sf[:, 19], color=c, marker="o", ms=4, label=str(d.date()))
    ax2[1].plot(st / s0, sf[2, :], color=c, lw=1.2, label=str(d.date()))
ax2[1].axvline(1.0, color="#30363d", linestyle="--", lw=1)
for ax, ttl, xl, yl in zip(ax2,
    ["ATM Term Structure Over Time","Volatility Skew (6m Maturity)"],
    ["Maturity (years)","Moneyness K/S₀"],
    ["ATM Implied Vol","Implied Vol (6m slice)"]
):
    ax.set_title(ttl, color="#c9d1d9"); ax.set_xlabel(xl)
    ax.set_ylabel(yl); ax.legend(fontsize=7); ax.grid(True)
plt.tight_layout()

# ── Figure 3: Strategy Dashboard ────────────────────────────────────────────
fig3 = plt.figure(figsize=(20, 12))
fig3.patch.set_facecolor("#0d1117")
fig3.suptitle("Multi-Strategy Volatility Dashboard", fontsize=14, color="#c9d1d9", y=0.99)
gs = gridspec.GridSpec(3, 4, figure=fig3, hspace=0.42, wspace=0.35)

ax = fig3.add_subplot(gs[0, :2]); ax.set_facecolor("#0d1117")
vp = spy[["ewma_vol","garch_vol","rv_21"]].iloc[WB:]
ax.plot(vp.index, vp["rv_21"],     color="#4a5568", lw=0.8, label="RV 21d")
ax.plot(vp.index, vp["ewma_vol"],  color=NEON_BLUE,   lw=1.2, label="EWMA")
ax.plot(vp.index, vp["garch_vol"], color=NEON_ORANGE, lw=1.2, label="GARCH(1,1)")
for rid, fc in [(0,"#1f4e2c"),(1,"#2d3f5e"),(2,"#5e1f1f")]:
    mask = spy["regime"].iloc[WB:] == rid
    ax.fill_between(vp.index, 0, vp["ewma_vol"].max()*1.1,
                    where=mask.values, alpha=0.12, color=fc, label=f"Regime {rid}")
ax.set_title("Vol Estimators + Regime Overlay", color="#c9d1d9", fontsize=9)
ax.legend(fontsize=7, ncol=3); ax.set_ylabel("Ann. Vol"); ax.grid(True)

ax = fig3.add_subplot(gs[0, 2:]); ax.set_facecolor("#0d1117")
vrp_p = spy["vrp"].iloc[WB:]
ax.plot(vrp_p.index, vrp_p, color=NEON_GREEN, lw=0.9, label="VRP")
ax.axhline(0, color="#30363d", lw=0.8)
ax.fill_between(vrp_p.index, vrp_p, 0, where=vrp_p > 0, alpha=0.25, color=NEON_GREEN)
ax.fill_between(vrp_p.index, vrp_p, 0, where=vrp_p < 0, alpha=0.25, color=NEON_ORANGE)
trig = bt[bt["cond_vrp"]==True]
ax.scatter(trig.index, spy.loc[trig.index,"vrp"], color=NEON_YELLOW, s=12, zorder=5, label="VRP trigger")
ax.set_title("Variance Risk Premium + Triggers", color="#c9d1d9", fontsize=9)
ax.legend(fontsize=7); ax.set_ylabel("VRP"); ax.grid(True)

ax = fig3.add_subplot(gs[1, :2]); ax.set_facecolor("#0d1117")
dp = spy["dispersion"].iloc[WB:].dropna()
ax.plot(dp.index, dp, color=NEON_PURPLE, lw=0.9)
trig_d = bt[bt["cond_disp"]==True]
ax.scatter(trig_d.index, spy.loc[trig_d.index,"dispersion"],
           color=NEON_YELLOW, s=12, zorder=5, label="Dispersion trigger")
ax.set_title("Dispersion Signal (Constituent − Index Vol)", color="#c9d1d9", fontsize=9)
ax.legend(fontsize=7); ax.set_ylabel("Dispersion"); ax.grid(True)

ax = fig3.add_subplot(gs[1, 2:]); ax.set_facecolor("#0d1117")
cum_cols = ["cum_pnl_exotic_z","cum_pnl_vrp","cum_pnl_regime",
            "cum_pnl_disp","cum_pnl_skew","cum_pnl_portfolio"]
lbls = ["Exotic Proxy","VRP","Regime Break","Dispersion","Skew Arb","Portfolio"]
pal  = [NEON_BLUE, NEON_GREEN, NEON_ORANGE, NEON_PURPLE, NEON_YELLOW, "#ff6ac1"]
for col, lb, clr in zip(cum_cols, lbls, pal):
    ax.plot(bt.index, bt[col], color=clr, lw=2.0 if "portfolio" in col else 1.0,
            label=lb, alpha=1.0 if "portfolio" in col else 0.75)
ax.axhline(0, color="#30363d", lw=0.6)
ax.set_title("Cumulative PnL (log-return units)", color="#c9d1d9", fontsize=9)
ax.legend(fontsize=7, ncol=2); ax.grid(True)

ax = fig3.add_subplot(gs[2, :2]); ax.set_facecolor("#0d1117")
rs_cols = ["rs_pnl_vrp","rs_pnl_regime","rs_pnl_disp","rs_pnl_skew","rs_pnl_portfolio"]
for col, lb, clr in zip(rs_cols, ["VRP","Regime","Dispersion","Skew Arb","Portfolio"],
                         [NEON_GREEN,NEON_ORANGE,NEON_PURPLE,NEON_YELLOW,"#ff6ac1"]):
    ax.plot(bt.index, bt[col], color=clr, lw=2.0 if "portfolio" in col else 0.9,
            label=lb, alpha=1.0 if "portfolio" in col else 0.7)
ax.axhline(0, color="#30363d", lw=0.8)
ax.set_title("Rolling 63-day Sharpe Ratio", color="#c9d1d9", fontsize=9)
ax.legend(fontsize=7, ncol=3); ax.set_ylabel("Sharpe"); ax.grid(True)

ax = fig3.add_subplot(gs[2, 2:]); ax.set_facecolor("#0d1117")
for col, lb, clr in zip(["pnl_vrp","pnl_regime","pnl_disp","pnl_skew","pnl_portfolio"],
                         ["VRP","Regime","Dispersion","Skew Arb","Portfolio"],
                         [NEON_GREEN,NEON_ORANGE,NEON_PURPLE,NEON_YELLOW,"#ff6ac1"]):
    ax.hist(bt[col].dropna(), bins=60, density=True, alpha=0.45, color=clr, label=lb)
ax.set_title("PnL Distributions (density)", color="#c9d1d9", fontsize=9)
ax.legend(fontsize=7, ncol=2); ax.set_xlabel("PnL (log-return units)"); ax.grid(True)


# ── Figure 4: Per-Regime Sharpe ──────────────────────────────────────────────
fig4, ax4 = plt.subplots(1, 3, figsize=(18, 5))
fig4.patch.set_facecolor("#0d1117")
fig4.suptitle("Per-Regime Strategy Sharpe Decomposition", fontsize=12, color="#c9d1d9")
pk   = ["pnl_vrp","pnl_regime","pnl_disp","pnl_skew"]
snms = ["VRP","Regime","Dispersion","Skew Arb"]
for ri, (rn, rc) in enumerate(zip(["Low","Medium","High"],[NEON_GREEN,NEON_YELLOW,NEON_ORANGE])):
    ax = ax4[ri]; ax.set_facecolor("#0d1117")
    sub = bt[bt["regime"] == ri]
    shs = [np.sqrt(252)*sub[p].dropna().mean()/(sub[p].dropna().std()+1e-9)
           if len(sub[p].dropna()) > 5 else 0.0 for p in pk]
    bars = ax.bar(snms, shs, color=[NEON_GREEN,NEON_ORANGE,NEON_PURPLE,NEON_YELLOW],
                  alpha=0.85, edgecolor="#21262d")
    ax.axhline(0, color="#c9d1d9", lw=0.6)
    for bar, val in zip(bars, shs):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.02*np.sign(val),
                f"{val:.2f}", ha="center", fontsize=8, color="#c9d1d9")
    ax.set_title(f"{rn} Vol Regime", color=rc, fontsize=10)
    ax.set_ylabel("Annualised Sharpe"); ax.grid(axis="y", alpha=0.4)
    ax.tick_params(axis="x", rotation=20, labelsize=8)
plt.tight_layout()

# ── Figure 5: Drawdowns ──────────────────────────────────────────────────────
fig5, ax5 = plt.subplots(figsize=(16, 5))
fig5.patch.set_facecolor("#0d1117"); ax5.set_facecolor("#0d1117")
fig5.suptitle("Strategy Drawdowns", fontsize=12, color="#c9d1d9")
for col, lb, clr in zip(pk+["pnl_portfolio"], snms+["Portfolio"],
                         [NEON_GREEN,NEON_ORANGE,NEON_PURPLE,NEON_YELLOW,"#ff6ac1"]):
    cum = bt[col].cumsum(); dd = cum - cum.cummax()
    ax5.fill_between(bt.index, dd, 0, alpha=0.25, color=clr)
    ax5.plot(bt.index, dd, color=clr, lw=0.9 if lb != "Portfolio" else 1.6, label=lb)
ax5.axhline(0, color="#30363d", lw=0.6)
ax5.set_ylabel("Drawdown"); ax5.legend(fontsize=8); ax5.grid(True)
plt.tight_layout()

print("\nDone.")

plt.show()

Running backtest …

===== STRATEGY PERFORMANCE =====
  Exotic Proxy (z-scored)   Sharpe=-0.090  MaxDD=-0.9827  Skew=+0.67  Kurt=94.06  Hit=49.58%
  VRP Mean-Reversion        Sharpe=-0.052  MaxDD=-0.4467  Skew=+0.76  Kurt=51.64  Hit=14.99%
  Vol Regime Breakout       Sharpe=+0.071  MaxDD=-0.2516  Skew=-2.38  Kurt=155.73  Hit=1.74%
  Conditional Dispersion    Sharpe=+0.366  MaxDD=-0.0551  Skew=-3.40  Kurt=103.32  Hit=3.03%
  GARCH Skew Arb            Sharpe=-0.247  MaxDD=-0.0463  Skew=-13.73  Kurt=421.97  Hit=1.24%
  EW Portfolio              Sharpe=-0.067  MaxDD=-0.2284  Skew=-0.22  Kurt=73.84  Hit=49.33%

Generating figures …

Done.
